In [1]:
import pandas as pd
import numpy as np
import kagglehub
import random
import torch
import torch.nn as nn
import os

from sklearn.model_selection import train_test_split
from sklearn.metrics import cohen_kappa_score

from transformers import AutoTokenizer, AutoModel
from torch.utils.data import Dataset, DataLoader

In [2]:
class cfg:
    competition = 'learning-agency-lab-automated-essay-scoring-2'
    checkpoint = 'microsoft/deberta-v3-base'
    tokenizer = None
    max_length = 512
    batch_size = 4
    epochs=10


In [3]:
def get_score(y_acts, y_preds):
    score = cohen_kappa_score(y_acts, y_preds, weights='quadratic')
    return score

In [4]:
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    
seed_everything(seed=42)

In [5]:
path = kagglehub.competition_download(cfg.competition)

In [6]:
train_df = pd.read_csv(f'{path}/train.csv')
test_df = pd.read_csv(f'{path}/test.csv')

In [7]:
if train_df['score'].min() == 1:
    train_df['score'] = train_df['score'] - 1

In [8]:
X_train, X_valid, y_train, y_valid = train_test_split(
                                train_df['full_text'], 
                                train_df['score'],
                                test_size=0.2,
                                stratify=train_df['score'],
                                random_state=42                       
                                )

In [9]:
train_df = pd.DataFrame({
    'full_text': X_train,
    'score': y_train
})
valid_df = pd.DataFrame({
    'full_text': X_valid,
    'score': y_valid
})

In [10]:
tokenizer = AutoTokenizer.from_pretrained(cfg.checkpoint)

In [11]:
class EssayDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=cfg.max_length):
        self.X = data['full_text']
        if 'score' in data.columns:
            self.y = data['score']
        else:
            self.y = None
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        tokens = self.tokenizer(
            text = self.X.iloc[idx],
            max_length=self.max_length,
            truncation = True,
            padding = 'max_length',
            return_tensors = 'pt'
        )
        item = {
            'input_ids': tokens['input_ids'].squeeze(0),
            'attention_mask': tokens['attention_mask'].squeeze(0),
        }
        if self.y is not None:
            item['labels'] = torch.tensor(self.y.iloc[idx], dtype=torch.long)
        return item


In [12]:
#MeanPooling code taken from online resources
class MeanPooling(nn.Module):
    def __init__(self):
        super(MeanPooling, self).__init__()

    def forward(self, last_hidden_state, attention_mask):
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
        sum_embeddings = torch.sum(last_hidden_state * input_mask_expanded, 1)
        sum_mask = input_mask_expanded.sum(1)
        sum_mask = torch.clamp(sum_mask, min=1e-9)
        mean_embeddings = sum_embeddings/sum_mask
        return mean_embeddings

In [13]:
class EssayModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = AutoModel.from_pretrained(cfg.checkpoint)
        self.mp = MeanPooling()
        self.classifier = nn.Linear(self.model.config.hidden_size, 6)

    def forward(self, input_ids, attention_mask):
        output = self.model(input_ids = input_ids, attention_mask = attention_mask)
        mp_output = self.mp(output.last_hidden_state, attention_mask)
        logits = self.classifier(mp_output)
        return logits


In [14]:
print(torch.backends.mps.is_available())
device = torch.device('mps')
#For CUDA:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
deberta_model = EssayModel().to(device)

True


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

[transformers] DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
DEBUG = True
if DEBUG:
    train_df_used = train_df.sample(frac=0.01, random_state=42)
    valid_df_used = valid_df.sample(frac=0.01, random_state=42)
else:
    train_df_used = train_df
    valid_df_used = valid_df
   
train_dataset = EssayDataset(train_df_used, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=cfg.batch_size, shuffle=True)
valid_dataset = EssayDataset(valid_df_used, tokenizer)
valid_loader = DataLoader(valid_dataset, batch_size=cfg.batch_size, shuffle=False)

test_dataset = EssayDataset(test_df, tokenizer)
test_loader = DataLoader(test_dataset, batch_size=cfg.batch_size, shuffle = False)

In [16]:
optimizer = torch.optim.AdamW(
    deberta_model.parameters(), 
    lr=1e-5,
    eps=1e-6,
    )
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

loss_fn = nn.CrossEntropyLoss()

#Store best QWK
best_kappa = -999

for epoch in range(cfg.epochs):
    print(f'----EPOCH {epoch+1}/{cfg.epochs}----')

    #Train
    deberta_model.train()
    train_loss = 0; train_correct = 0; train_total = 0

    for batch in train_loader:
        optimizer.zero_grad()
        labels = batch['labels'].to(device)
        logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        loss = loss_fn(logits, labels)
        
        preds = torch.argmax(logits, dim=1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)
        train_loss += loss.item()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(
            deberta_model.parameters(),
            max_norm=1.0
        )
        optimizer.step()

    avg_train_loss = train_loss/len(train_loader)
    train_accuracy = train_correct / train_total


    #Valid
    deberta_model.eval()
    valid_loss = 0; valid_correct = 0; valid_total = 0
    all_labels = []; all_preds = []

    with torch.no_grad():
        for batch in valid_loader:
            labels = batch['labels'].to(device)
            logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
            loss = loss_fn(logits, labels)
            valid_loss += loss.item()
            preds = torch.argmax(logits, dim=1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            valid_correct += (preds==labels).sum().item()
            valid_total += labels.size(0)

    avg_valid_loss = valid_loss/len(valid_loader)
    valid_accuracy = valid_correct/valid_total

    kappa = get_score(all_labels, all_preds)
    if kappa > best_kappa:
        best_kappa = kappa
        torch.save(
            deberta_model.state_dict(),
            'best_deberta_model.pth'
        )
        print(f'Saved new-best model: QWK {best_kappa:.4f}')

    scheduler.step()

    #Summary
    print(f'Train Loss: {avg_train_loss:.4f}')
    print(f'Train Accuracy: {train_accuracy:.4f}')
    print(f'Valid Loss: {avg_valid_loss:.4f}')
    print(f'Valid Accuracy: {valid_accuracy:.4f}')
    print(f'Current Epoch QWK: {kappa:.4f}')
    print(f'Best QWK: {best_kappa:.4f}')
    print(f'LR: {scheduler.get_last_lr()[0]:.2e}\n')

----EPOCH 1/10----
Saved new-best model: QWK 0.0000
Train Loss: 1.5968
Train Accuracy: 0.3261
Valid Loss: 1.3476
Valid Accuracy: 0.4000
Current Epoch QWK: 0.0000
Best QWK: 0.0000
LR: 9.76e-06

----EPOCH 2/10----
Train Loss: 1.4582
Train Accuracy: 0.3406
Valid Loss: 1.3898
Valid Accuracy: 0.4000
Current Epoch QWK: 0.0000
Best QWK: 0.0000
LR: 9.05e-06

----EPOCH 3/10----
Train Loss: 1.4317
Train Accuracy: 0.3043
Valid Loss: 1.3176
Valid Accuracy: 0.4000
Current Epoch QWK: 0.0000
Best QWK: 0.0000
LR: 7.94e-06

----EPOCH 4/10----
Train Loss: 1.4264
Train Accuracy: 0.3768
Valid Loss: 1.3184
Valid Accuracy: 0.4000
Current Epoch QWK: 0.0000
Best QWK: 0.0000
LR: 6.55e-06

----EPOCH 5/10----


KeyboardInterrupt: 

In [ ]:
#Test
#Load best model
deberta_model.load_state_dict(
    torch.load('best_deberta_model.pth')
)
deberta_model.eval()
test_preds = []

with torch.no_grad():
    for batch in test_loader:
        logits = deberta_model(batch['input_ids'].to(device), batch['attention_mask'].to(device))
        preds = torch.argmax(logits, dim=1)
        test_preds.extend(preds.cpu().numpy())

#Add 1 back to scores
test_preds = [pred+1 for pred in test_preds]

#Submission
submission = pd.DataFrame({
    'essay_id': test_df['essay_id'],
    'score': test_preds
})
submission.to_csv('submission.csv', index=False)